# Regional-mean surface-temperature budget

Monthly regional-mean TS budget terms and per-model/season CSVs for one region.
Set `PROJECT` and `REGION` in the configuration cell; region settings come from
`config/regions.json`. Ported from `v3_polar_analysis/<region>/plot_mean_TS_budget.ipynb`.

In [ ]:
from pathlib import Path
import sys

# Make this package's scripts/ importable (exp_info, regions, data_utils, ...).
PROJECT_ROOT = next(
    (path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (path / "scripts" / "exp_info.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start this notebook from inside the polar_analysis package.")

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

In [ ]:
# Parameters: directories and settings for this run. Edit them for another
# machine or experiment, or override with papermill (-p NAME VALUE).
REGION    = "Antarctic"  # Arctic, Greenland, Antarctic, Atlantic (config/regions.json)
CLIMO_DIR = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3_polar_analysis/data/climo"  # input climatologies
DIAG_DIR  = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3_polar_analysis/diag_data"  # output diagnostic data (<REGION>/ is added)
FIG_DIR   = "/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/v3_polar_analysis/figures/budget"  # output figures (<REGION>/ is added)

In [ ]:
from __future__ import annotations

import os
import glob
import re
import json
import time
import datetime
from typing import Dict, Tuple, Optional
import pandas as pd 
import xarray as xr

import unicodedata
import pprint, shutil

try:
    # xCDAT for CF bounds utilities and lon handling
    from xcdat import open_dataset as xcdat_open
    _HAVE_XCDAT = True
except Exception:
    xcdat_open = xr.open_dataset
    _HAVE_XCDAT = False

try:
    import xskillscore as xs
    _HAVE_XS = True
except Exception:
    _HAVE_XS = False

# optional: MetPy smoothing (safe fallback if not installed)
try:
    from metpy.calc import smooth_n_point as _smooth_n_point
    _HAS_METPY = True
except Exception:
    _HAS_METPY = False
    def _smooth_n_point(arr, n=9, passes=2):
        return arr  # no-op fallback
    
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as matcolors
import matplotlib.path as matpath
import matplotlib.ticker as mticker
import matplotlib.contour as mcontour
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from collections.abc import Mapping

from matplotlib.offsetbox import AnchoredText

# Local imports
from exp_info import RUN_CATALOG, REGION_CATALOG, VARIABLE_CATALOG, Budget_CATALOG


In [ ]:
class ExpDataReader:
    def __init__(
        self,
        data_dir: str,
        regnam: str,
        dtype: str,
        *,
        run_catalog: Optional[Dict[str, Dict[str, str]]] = None,
        region_catalog: Optional[Dict[str, Tuple[Tuple[float, float], Tuple[float, float]]]] = None,
        land_mask_ref: str = "MODIS.observation",
        land_mask_var: str = "TS",
        debug_key: Optional[str] = None,
        verbose: bool = False,
    ):
        self.data_dir = os.path.abspath(data_dir)
        self.regnam = regnam
        self.dtype = dtype
        
        # copy to avoid accidental mutation of module-level defaults
        self.run_catalog = dict(RUN_CATALOG) if run_catalog is None else dict(run_catalog)
        self.region_catalog = dict(REGION_CATALOG) if region_catalog is None else dict(region_catalog)
        
        self.land_mask_var = land_mask_var
        self.land_mask_ref = land_mask_ref
        self.debug_key = debug_key
        self.verbose = verbose

        (lat_min, lat_max), (lon_min, lon_max) = self._region_bounds(self.regnam)
        lon_min, lon_max = self._normalize_region_lon_bounds(lon_min, lon_max)
        self.lat_slice = slice(lat_min, lat_max)
        self.lon_bounds = (lon_min, lon_max)

        self.exp_dict: Dict[str, xr.Dataset] = {}
        self.land_mask: Optional[xr.DataArray] = None

    def _meta_field(self, meta, field: str):
        """Return run metadata field from object or dict."""
        if hasattr(meta, field):
            return getattr(meta, field)
        if isinstance(meta, dict) and field in meta:
            return meta[field]
        raise KeyError(f"Run meta missing '{field}': {field} not found in {meta!r}")

    def _region_bounds(self, regnam: str):
        """Return ((lat_min, lat_max), (lon_min, lon_max)) from Region object/dict/tuple."""
        try:
            region = self.region_catalog[regnam]
        except KeyError:
            raise ValueError(f"Region '{regnam}' not found. Available: {list(self.region_catalog)}")

        # Region dataclass / object
        if hasattr(region, "lat_range") and hasattr(region, "lon_range"):
            (lat_min, lat_max) = region.lat_range
            (lon_min, lon_max) = region.lon_range
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))

        # Dict style
        if isinstance(region, dict) and "lat_range" in region and "lon_range" in region:
            (lat_min, lat_max) = region["lat_range"]
            (lon_min, lon_max) = region["lon_range"]
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))

        # Tuple/list style: ((lat_min, lat_max), (lon_min, lon_max))
        try:
            (lat_min, lat_max), (lon_min, lon_max) = region
            return (float(lat_min), float(lat_max)), (float(lon_min), float(lon_max))
        except Exception:
            raise TypeError(
                "REGION_CATALOG entry must be a Region with lat_range/lon_range, "
                "a dict with those keys, or ((lat_min,lat_max),(lon_min,lon_max))."
            )
        
    # --- longitude handling ---
    @staticmethod
    def _is_0_360(lons) -> bool:
        """Heuristic: grid in [0, 360) if min(lon) >= 0 and max(lon) > 180."""
        a = np.asarray(lons)
        return np.nanmin(a) >= 0.0 and np.nanmax(a) > 180.0

    @staticmethod
    def _wrap_to_180(lon_vals):
        lon_vals = (lon_vals + 180.0) % 360.0 - 180.0
        return lon_vals

    def _normalize_longitude(self, ds: xr.Dataset) -> xr.Dataset:
        """Ensure ds.lon is in [-180, 180] and sorted."""
        lon_name = "lon" if "lon" in ds.coords else ("longitude" if "longitude" in ds.coords else None)
        if lon_name is None:
            return ds  # nothing to do

        lons = ds[lon_name].values
        if self._is_0_360(lons):
            ds = ds.assign_coords({lon_name: self._wrap_to_180(ds[lon_name])}).sortby(lon_name)

        # For consistency rename to 'lon' if it's 'longitude'
        if lon_name == "longitude":
            ds = ds.rename({lon_name: "lon"})
        return ds

    @staticmethod
    def _normalize_region_lon_bounds(lon_min: float, lon_max: float):
        """Ensure region bounds use the [-180, 180] convention.
        If already in range, return as-is; otherwise convert.
        """
        if (lon_min < -180 or lon_min > 180) and (lon_max >= -180 and lon_max <= 180):
            raise ValueError("Mixed longitude conventions in region bounds")
            
        if -180.0 <= lon_min <= 180.0 and -180.0 <= lon_max <= 180.0:
            return lon_min, lon_max
        lon_min = (lon_min + 180.0) % 360.0 - 180.0
        lon_max = (lon_max + 180.0) % 360.0 - 180.0
        return lon_min, lon_max

    def _select_region_and_bounds(self, ds: xr.Dataset) -> xr.Dataset:
        """Subset by lat and lon, handling dateline crossing for lon bounds."""
        ds = self._maybe_add_bounds(ds)
        ds = ds.sel(lat=self.lat_slice) if "lat" in ds.coords else ds

        if "lon" not in ds.coords:
            return ds

        lon_min, lon_max = self.lon_bounds

        if lon_min <= lon_max:
            # simple case
            return ds.sel(lon=slice(lon_min, lon_max))
        else:
            # crosses the dateline, e.g., lon_min=170, lon_max=-150
            left = ds.sel(lon=slice(lon_min, 180))
            right = ds.sel(lon=slice(-180, lon_max))
            # Align before concat in case of float precision differences
            right = right.assign_coords(lon=self._wrap_to_180(right["lon"]))
            return xr.concat([left, right], dim="lon").sortby("lon")

    def _maybe_add_bounds(self, ds: xr.Dataset) -> xr.Dataset:
        if _HAVE_XCDAT:
            try:
                ds = ds.bounds.add_missing_bounds()
            except Exception:
                pass
        return ds

    def _sync_time_to_reference(self):
        ref_time = self.exp_dict[self.land_mask_ref]["time"]
        for k, ds in self.exp_dict.items():
            if k == self.land_mask_ref:
                continue
            if "time" in ds.coords:
                if len(ds["time"]) != len(ref_time):
                    raise ValueError(
                        f"[time sync] Mismatch for {k}: {len(ds['time'])} vs {len(ref_time)}"
                    )
                self.exp_dict[k] = ds.assign_coords(time=ref_time)

    def _debug_print(self, key, path, lats0=None, lons0=None):
        if key != self.debug_key:
            return
        print(f"[DEBUG] {key}")
        if lats0 is not None:
            print("  lats0:", np.asarray(lats0).min(), "→", np.asarray(lats0).max())
        if lons0 is not None:
            print("  lons0:", np.asarray(lons0).min(), "→", np.asarray(lons0).max())
        print("  period:", self.run_catalog[key]["period"])
        print("  path:", path)
        print("  lon(post):", self.exp_dict[key]["lon"].min().item(), "→", self.exp_dict[key]["lon"].max().item())

    def _normalize_period_str(self, s: str) -> str:
        _DASHES = {
            "\u2010": "-",  # hyphen
            "\u2011": "-",  # non-breaking hyphen
            "\u2012": "-",  # figure dash
            "\u2013": "-",  # en dash
            "\u2014": "-",  # em dash
            "\u2212": "-",  # minus sign
        }
        # NFKC squashes some oddities; then map all dash-like chars to ASCII '-'
        s = unicodedata.normalize("NFKC", s)
        return s.translate(str.maketrans(_DASHES)).strip()
    
    def _gen_land_mask(self, var: Optional[str] = None) -> xr.DataArray:
        var = var or self.land_mask_var

        # Always prefer land fraction vars
        land_vars = ("sftlf", "LANDFRAC", "landfrac", "land_mask")
        if var not in land_vars:
            print(f"[WARN] Using '{var}' for land mask, may misclassify oceans.")

        # reference file path (normalize Unicode dashes)
        ref_meta = self.run_catalog[self.land_mask_ref]
        ref_period_raw = self._meta_field(ref_meta, "period")
        ref_period = self._normalize_period_str(ref_period_raw)

        refpath = os.path.join(self.data_dir, f"{self.land_mask_ref}.{self.dtype}.{ref_period}.nc")
        if not os.path.exists(refpath):
            raise FileNotFoundError(f"[landmask] reference file not found: {refpath}")

        # open and normalize longitudes like other datasets
        refds = xcdat_open(refpath, decode_times=True)
        refds = self._normalize_longitude(refds)

        # subset to region (handles dateline crossing)
        refds = self._select_region_and_bounds(refds)

        # ensure bounds if xCDAT is available (kept for consistency with your pipeline)
        refds = self._maybe_add_bounds(refds)

        # choose a source field for mask
        if var in refds.data_vars:
            src = refds[var]
            # reproduce your original behavior (> 0 → 1, else 0)
            mask = xr.where(src > 0, 1, 0)
        else:
            # fallbacks: try common land fraction names, else use finite values
            for alt in ("sftlf", "LANDFRAC", "landfrac", "land_mask"):
                if alt in refds.data_vars:
                    mask = xr.where(refds[alt] > 0, 1, 0)
                    break
            else:
                # last resort: any finite values in the reference file define valid mask
                # pick the first 2D var we find
                two_d_vars = [v for v in refds.data_vars if set(refds[v].dims) >= {"lat", "lon"}]
                if not two_d_vars:
                    raise ValueError("[landmask] could not infer a 2D variable to base a mask on.")
                src = refds[two_d_vars[0]]
                mask = xr.where(np.isfinite(src), 1, 0)

        # keep only lat/lon dims and attach a helpful name
        mask = mask.squeeze()
        # if var carried time, drop it now (mask is purely horizontal)
        drop_dims = [d for d in mask.dims if d not in ("lat", "lon")]
        if drop_dims:
            mask = mask.isel({d: 0 for d in drop_dims})
        mask = mask.astype(np.int8).rename("land_mask")
        return mask

    # --- main API ---
    def read(self, debug=False):
        for exp_key, meta in self.run_catalog.items():
            raw_period = self._meta_field(meta, "period")
            period = self._normalize_period_str(raw_period)
            nc_path = os.path.join(self.data_dir, f"{exp_key}.{self.dtype}.{period}.nc")
            if self.verbose:
                print(f"Loading: {nc_path}")

            if not os.path.exists(nc_path):
                raise FileNotFoundError(f"File not found for run '{exp_key}': {nc_path}")

            ds = xcdat_open(nc_path, decode_times=True)

            lats0 = ds["lat"].values if "lat" in ds.coords else None
            lons0 = (ds["lon"].values if "lon" in ds.coords
                     else (ds["longitude"].values if "longitude" in ds.coords else None))

            ds = self._normalize_longitude(ds)
            ds = self._select_region_and_bounds(ds)

            self.exp_dict[exp_key] = ds
            if debug:
                self._debug_print(exp_key, nc_path, lats0, lons0)

        if self.land_mask_ref not in self.exp_dict:
            raise KeyError(f"Reference '{self.land_mask_ref}' not loaded (after reading).")

        # Align time to reference
        self._sync_time_to_reference()

        # Build land mask using the reference run's period
        ref_meta = self.run_catalog[self.land_mask_ref]
        ref_period_raw = self._meta_field(ref_meta, "period")
        ref_period = self._normalize_period_str(ref_period_raw)   # <-- normalize dashes
        self.land_mask = self._gen_land_mask(self.land_mask_var)
        return self.run_catalog, self.exp_dict, self.land_mask
    

In [ ]:
class RegionalMeanBudget:
    """
    Regional-mean energy budget analysis.

    Computes and plots regional-mean time series of budget terms
    (e.g., TS_ALL, TS_TURB, TS_Qn, TS_CRE, TS_SAF, TS_nSAF, TS_DLW),
    averaged over a user-specified region with optional land/sea mask
    and day weighting.

    Key methods:
      - regional_budget_timeseries()
      - regional_seasonal_means()
      - plot_regional_timeseries()

    Output filename convention:
      regional_budget_ts_{budget_type}_{term}_{region}.pdf
    """

    def __init__(
        self,
        run_dict,
        df,
        lmsk,
        regnam,
        model_list,
        *,
        budget_vars=None,        # list[str] of budget terms to compute/plot
        term_specs=None,         # optional per-term specs (unused here but kept for API symmetry)
        budget_type="best_obs",
        region_bounds=None,      # ((lat_min,lat_max),(lon_min,lon_max)) or None
        debug=False,
        budget_fn=None,          # optional external override for budget derivation
    ):
        self.run_dict = run_dict
        self.df = df
        self.lmsk = lmsk
        self.term_specs = term_specs or {}
        self.regnam = regnam
        self.model_list = list(model_list)
        self.budget_type = budget_type
        self.debug = bool(debug)
        self.budget_fn = budget_fn

        # Which terms to compute?
        if budget_vars is not None and len(budget_vars) > 0:
            self.plot_terms = list(budget_vars)
        else:
            self.plot_terms = list(self.term_specs.keys()) if self.term_specs else ["TS_ALL"]

        # region ranges
        self.lat_range = (-90.0, 90.0)
        self.lon_range = (-180.0, 180.0)
        if isinstance(region_bounds, (list, tuple)) and len(region_bounds) == 2:
            self.lat_range, self.lon_range = region_bounds

    # ---------- helpers ----------
    @staticmethod
    def _generate_coslat_weight(da):
        lat = da["lat"] if "lat" in da.coords else da
        return xr.DataArray(np.cos(np.deg2rad(lat)), dims=["lat"], coords={"lat": lat})

    @staticmethod
    def _meta_get(meta, key, default=None):
        try:
            return meta.get(key, default)
        except Exception:
            return getattr(meta, key, default)

    @staticmethod
    def _restrict_lonlat(da, lat_range, lon_range):
        if "lon" in da.coords:
            lon = da["lon"]
            if float(lon.max()) > 180.0:
                new_lon = ((lon + 180.0) % 360.0) - 180.0
                da = da.assign_coords(lon=new_lon).sortby("lon")
        lat0, lat1 = lat_range
        lon0, lon1 = lon_range
        if "lat" in da.coords:
            da = da.sel(lat=slice(lat0, lat1))
        if "lon" in da.coords:
            if lon0 <= lon1:
                da = da.sel(lon=slice(lon0, lon1))
            else:
                da = xr.concat(
                    [da.sel(lon=slice(lon0, 180)), da.sel(lon=slice(-180, lon1))],
                    dim="lon",
                )
        return da

    @staticmethod
    def _to_mask_like(lmsk, like_da: xr.DataArray) -> xr.DataArray:
        if isinstance(lmsk, xr.Dataset):
            for key in ("mask", "landmask", "sftlf"):
                if key in lmsk:
                    lmsk = lmsk[key]
                    break
            else:
                raise ValueError("Mask Dataset provided but no ['mask','landmask','sftlf'] var found.")
        if isinstance(lmsk, xr.DataArray):
            m = lmsk
            if "lat" in like_da.coords and "lat" in m.coords:
                m = m.sel(lat=like_da.lat, method="nearest")
            if "lon" in like_da.coords and "lon" in m.coords:
                m = m.sel(lon=like_da.lon, method="nearest")
            m = m.astype("float64")
        else:
            m = xr.DataArray(
                np.asarray(lmsk),
                dims=["lat", "lon"],
                coords={"lat": like_da.lat, "lon": like_da.lon},
            ).astype("float64")
        return xr.where(m > 0.5, 1.0, 0.0)

    @staticmethod
    def _budget_cmap():
        color_list4 = ["darkblue", "lightblue", "white", "lightyellow", "orange", "firebrick"]
        nodes = [0.0, 0.3, 0.5, 0.500001, 0.8, 1.0]  # strictly increasing (matplotlib >= 3.11)
        return matcolors.LinearSegmentedColormap.from_list("mybluewhitered", list(zip(nodes, color_list4)))

    # ---------- internal budget derivation ----------
    @staticmethod
    def _ensure_kelvin(ts):
        try:
            mx = float(ts.max().values)
        except Exception:
            mx = np.nan
        return ts + 273.15 if (np.isfinite(mx) and mx < 150.0) else ts

    @staticmethod
    def _align(*das):
        out = xr.align(*das, join="inner")
        return tuple(a.astype("float64") for a in out)

    def _diag(self, model: str, name: str, da: xr.DataArray):
        if not self.debug:
            return
        v = da
        try:
            if hasattr(v.data, "compute"):
                v = v.compute()
        except Exception:
            pass
        arr = v.values
        vmin = np.nanmin(arr)
        vmax = np.nanmax(arr)
        vmean = np.nanmean(arr)
        nvalid = np.isfinite(arr).sum()
        print(f"[diag] {model:>18s} {name:8s} | min={vmin: .3f}, max={vmax: .3f}, mean={vmean: .3f}, nvalid={nvalid}")

    def _has(self, ds: xr.Dataset, name: str) -> bool:
        try:
            return name in ds
        except Exception:
            return False

    def _pick(self, ds: xr.Dataset, *names: str) -> xr.DataArray:
        for nm in names:
            if self._has(ds, nm):
                return ds[nm]
        raise KeyError(f"None of {names} found in dataset.")
        
    def _albedo_from_clear(self, ds: xr.Dataset):
        if self._has(ds, "ALBEDOC_SRF"):
            return ds["ALBEDOC_SRF"].astype("float64").clip(0.0, 1.0)
        fsnsc = self._pick(ds, "FSNSC").astype("float64")
        fsdsc = self._pick(ds, "FSDSC").astype("float64")
        ratio = xr.where(fsdsc == 0.0, 0.0, fsnsc / fsdsc)
        return (1.0 - ratio).clip(0.0, 1.0)

    def _derive_flux_anomalies_internal(self):
        """
        Build per-model flux fields used in the budget.

        Returns dict[model] -> xr.Dataset with:
          - Model absolute fluxes:   <NAME>_m
          - Reference absolute:      <NAME>_r
          - Anomalies:               d<NAME>  (or *_flux for SAF/nSAF)

        Units:
          - Fluxes: W m^-2
          - Albedo: unitless [0,1]

        Variables (if present in input):
          SHFLX, LHFLX, FSNS, FLNS, Qn, SWCF_SRF, LWCF_SRF, CRE,
          FLDS (prefers FLDSC), FSDSC, FSNSC, ALBEDO,
          SAF_flux, nSAF_flux.
        """
        if self.budget_type == "all_merra":
            refts = reftur = refrad = "MERRA2.analysis"
        elif self.budget_type == "best_obs":
            refts = "MODIS.observation"
            refrad = "CERES_EBAF.observation"
            reftur = "MERRA2.analysis"
        else:
            raise ValueError(f"Unknown budget_type: {self.budget_type}")

        # Reference albedo and insolation
        ts_r = self._ensure_kelvin(self.df[refts]["TS"]).astype("float64")
        albedo_r = self._albedo_from_clear(self.df[refrad])
        fsdsc_r  = self._pick(self.df[refrad], "FSDSC").astype("float64")

        out = {}
        for model in self.model_list:
            ds_out = xr.Dataset()
            # --- surface temperature 
            ts_m = self._ensure_kelvin(self.df[model]["TS"])
            ds_out["TS_m"] = ts_m; ds_out["TS_r"] = ts_r; ds_out["dTS"] = ts_m - ts_r
            
            # --- turbulent fluxes
            sh_m  = self._pick(self.df[model],  "SHFLX").astype("float64")
            lh_m  = self._pick(self.df[model],  "LHFLX").astype("float64")
            sh_r  = self._pick(self.df[reftur], "SHFLX").astype("float64")
            lh_r  = self._pick(self.df[reftur], "LHFLX").astype("float64")
            sh_m, sh_r = self._align(sh_m, sh_r)
            lh_m, lh_r = self._align(lh_m, lh_r)
            ds_out["SHFLX_m"] = sh_m; ds_out["SHFLX_r"] = sh_r; ds_out["dSH"] = sh_m - sh_r
            ds_out["LHFLX_m"] = lh_m; ds_out["LHFLX_r"] = lh_r; ds_out["dLH"] = lh_m - lh_r

            # --- net SW/LW fluxes
            fsns_m = self._pick(self.df[model],  "FSNS").astype("float64")
            flns_m = self._pick(self.df[model],  "FLNS").astype("float64")
            fsns_r = self._pick(self.df[refrad], "FSNS").astype("float64")
            flns_r = self._pick(self.df[refrad], "FLNS").astype("float64")
            fsns_m, fsns_r = self._align(fsns_m, fsns_r)
            flns_m, flns_r = self._align(flns_m, flns_r)
            ds_out["FSNS_m"] = fsns_m; ds_out["FSNS_r"] = fsns_r; ds_out["dFSNS"] = fsns_m - fsns_r
            ds_out["FLNS_m"] = flns_m; ds_out["FLNS_r"] = flns_r; ds_out["dFLNS"] = flns_m - flns_r

            # --- combined Qn
            qn_m = -fsns_m + flns_m + sh_m + lh_m
            qn_r = -fsns_r + flns_r + sh_r + lh_r
            ds_out["Qn_m"] = qn_m; ds_out["Qn_r"] = qn_r; ds_out["dQn"] = qn_m - qn_r

            # --- CRE
            swcf_m = self._pick(self.df[model],  "SWCF_SRF").astype("float64")
            lwcf_m = self._pick(self.df[model],  "LWCF_SRF").astype("float64")
            swcf_r = self._pick(self.df[refrad], "SWCF_SRF").astype("float64")
            lwcf_r = self._pick(self.df[refrad], "LWCF_SRF").astype("float64")
            swcf_m, swcf_r = self._align(swcf_m, swcf_r)
            lwcf_m, lwcf_r = self._align(lwcf_m, lwcf_r)
            cre_m = swcf_m + lwcf_m; cre_r = swcf_r + lwcf_r
            ds_out["SWCF_SRF_m"] = swcf_m; ds_out["SWCF_SRF_r"] = swcf_r; ds_out["dSWCF"] = swcf_m - swcf_r
            ds_out["LWCF_SRF_m"] = lwcf_m; ds_out["LWCF_SRF_r"] = lwcf_r; ds_out["dLWCF"] = lwcf_m - lwcf_r
            ds_out["CRE_m"]      = cre_m;  ds_out["CRE_r"]      = cre_r;  ds_out["dCRE"]  = cre_m - cre_r

            # --- downward LW (clear if available)
            if self._has(self.df[model], "FLDSC"):  flds_m = self._pick(self.df[model],  "FLDSC").astype("float64")
            else:                                   flds_m = self._pick(self.df[model],  "FLDS").astype("float64")
            if self._has(self.df[refrad], "FLDSC"): flds_r = self._pick(self.df[refrad], "FLDSC").astype("float64")
            else:                                   flds_r = self._pick(self.df[refrad], "FLDS").astype("float64")
            flds_m, flds_r = self._align(flds_m, flds_r)
            ds_out["FLDSC_m"] = flds_m; ds_out["FLDSC_r"] = flds_r; ds_out["dFLDSC"] = flds_m - flds_r

            # --- clear-sky insolation + albedo
            fsdsc_m = self._pick(self.df[model],  "FSDSC").astype("float64")
            fsdsc_m, fsdsc_r, albr = self._align(fsdsc_m, fsdsc_r, albedo_r)
            albedo_m = self._albedo_from_clear(self.df[model])
            albedo_m, albr = self._align(albedo_m, albr)
            ds_out["FSDSC_m"] = fsdsc_m; ds_out["FSDSC_r"] = fsdsc_r
            ds_out["ALBEDO_m"] = albedo_m; ds_out["ALBEDO_r"] = albr

            # --- SAF contributions
            ds_out["SAF_flux"]  = -(albedo_m - albr) * fsdsc_r
            ds_out["nSAF_flux"] = (1.0 - albr) * (fsdsc_m - fsdsc_r)

            out[model] = ds_out

        return out

    def _derive_energy_budget_internal(self):
        # choose reference datasets
        if self.budget_type == "all_merra":
            refts = reftur = refrad = "MERRA2.analysis"
        elif self.budget_type == "best_obs":
            refts = "MODIS.observation"
            refrad = "CERES_EBAF.observation"
            reftur = "MERRA2.analysis"
        else:
            raise ValueError(f"Unknown budget_type: {self.budget_type}")

        sigma = 5.67e-8
        ts_ref = self._ensure_kelvin(self.df[refts]["TS"]).astype("float64")
        denom = 4.0 * sigma * (ts_ref ** 3)
        factor_ref = 1.0 / xr.where(denom == 0.0, np.nan, denom)

        # helper to align factor to a target array's coords
        def _fac(target):
            fr, _ = self._align(factor_ref, target)
            return fr

        if self.debug:
            print(f"reference data: {refts}, {refrad}, {reftur}")

        # --- albedo reference (prefer direct ALBEDOC_SRF) ---
        if self._has(self.df[refrad], "ALBEDOC_SRF"):
            albedom = self.df[refrad]["ALBEDOC_SRF"].astype("float64").clip(0.0, 1.0)
        else:
            fsnsc = self._pick(self.df[refrad], "FSNSC").astype("float64")
            fsdsc = self._pick(self.df[refrad], "FSDSC").astype("float64")
            ratio = xr.where(fsdsc == 0.0, 0.0, fsnsc / fsdsc)
            albedom = (1.0 - ratio).clip(0.0, 1.0)

        budget_dict = {}
        for model in self.model_list:
            outds = self.df[model].copy()
            sum_requested = ("TS_Sum" in self.plot_terms)

            for term in self.plot_terms:
                if term == "TS_Sum":
                    continue

                if term == "TS_ALL":
                    ts_r, ts_m = self._align(ts_ref, self._ensure_kelvin(self.df[model]["TS"]))
                    outds["TS_ALL"] = ts_m - ts_r
                    self._diag(model, "TS_ALL", outds["TS_ALL"])

                elif term == "TS_TURB":
                    # positive-down convention: Q_turb = SH + LH
                    sh, lh, shr, lhr = self._align(
                        self._pick(self.df[model], "SHFLX"),
                        self._pick(self.df[model], "LHFLX"),
                        self._pick(self.df[reftur], "SHFLX"),
                        self._pick(self.df[reftur], "LHFLX"),
                    )
                    f = _fac(sh)
                    outds["TS_TURB"] = -((sh + lh) * f - (shr + lhr) * f)
                    self._diag(model, "TS_TURB", outds["TS_TURB"])

                elif term == "TS_Qn":
                    # positive-down: Qn = FSNS + FLNS + SH + LH
                    fs, fl, sh, lh, fsr, flr, shr, lhr = self._align(
                        self._pick(self.df[model], "FSNS"),
                        self._pick(self.df[model], "FLNS"),
                        self._pick(self.df[model], "SHFLX"),
                        self._pick(self.df[model], "LHFLX"),
                        self._pick(self.df[refrad], "FSNS"),
                        self._pick(self.df[refrad], "FLNS"),
                        self._pick(self.df[reftur], "SHFLX"),
                        self._pick(self.df[reftur], "LHFLX"),
                    )
                    qn_mod = -fs + fl + sh + lh
                    qn_ref = -fsr + flr + shr + lhr
                    f = _fac(qn_mod)
                    outds["TS_Qn"] = qn_mod * f - qn_ref * f
                    self._diag(model, "TS_Qn", outds["TS_Qn"])

                elif term == "TS_CRE":
                    swc, lwc, swcr, lwcr = self._align(
                        self._pick(self.df[model], "SWCF_SRF"),
                        self._pick(self.df[model], "LWCF_SRF"),
                        self._pick(self.df[refrad], "SWCF_SRF"),
                        self._pick(self.df[refrad], "LWCF_SRF"),
                    )
                    cre_m = swc + lwc
                    cre_r = swcr + lwcr
                    f = _fac(cre_m)
                    outds["TS_CRE"] = cre_m * f - cre_r * f
                    self._diag(model, "TS_CRE", outds["TS_CRE"])

                elif term == "TS_SAF":
                    # albedo change effect at clear-sky: -(Δalpha)*FSDSC_ref
                    if self._has(self.df[model], "ALBEDOC_SRF"):
                        alb_m = self.df[model]["ALBEDOC_SRF"].astype("float64").clip(0.0, 1.0)
                    else:
                        fsnsc_m = self._pick(self.df[model], "FSNSC").astype("float64")
                        fsdsc_m = self._pick(self.df[model], "FSDSC").astype("float64")
                        alb_m = (1.0 - xr.where(fsdsc_m == 0.0, 0.0, fsnsc_m / fsdsc_m)).clip(0.0, 1.0)

                    alb_m, alb_r, fsdsc_r = self._align(alb_m, albedom, self._pick(self.df[refrad], "FSDSC"))
                    saf = -(alb_m - alb_r) * fsdsc_r
                    f = _fac(saf)
                    outds["TS_SAF"] = saf * f
                    self._diag(model, "TS_SAF", outds["TS_SAF"])

                elif term == "TS_nSAF":
                    # insolation-driven clear-sky SW absorbed change holding alpha at ref
                    fsdsc_m, fsdsc_r, alb_r = self._align(
                        self._pick(self.df[model], "FSDSC"),
                        self._pick(self.df[refrad], "FSDSC"),
                        albedom,
                    )
                    nsaf = -(1.0 - alb_r) * (fsdsc_m - fsdsc_r)
                    f = _fac(nsaf)
                    outds["TS_nSAF"] = nsaf * f
                    self._diag(model, "TS_nSAF", outds["TS_nSAF"])

                elif term == "TS_DLW":
                    # downward LW (clear/all-sky fallback)
                    if self._has(self.df[model], "FLDSC"):
                        fldsc_m = self._pick(self.df[model], "FLDSC")
                    elif self._has(self.df[model], "FLDS"):
                        fldsc_m = self._pick(self.df[model], "FLDS")
                    else:
                        raise KeyError("No downward LW flux (FLDSC/FLDS) in model dataset")

                    if self._has(self.df[refrad], "FLDSC"):
                        fldsc_r = self._pick(self.df[refrad], "FLDSC")
                    elif self._has(self.df[refrad], "FLDS"):
                        fldsc_r = self._pick(self.df[refrad], "FLDS")
                    else:
                        raise KeyError(f"No downward LW flux (FLDSC/FLDS) in {refrad}")

                    fldsc_m, fldsc_r = self._align(fldsc_m, fldsc_r)
                    f = _fac(fldsc_m)
                    outds["TS_DLW"] = fldsc_m * f - fldsc_r * f
                    self._diag(model, "TS_DLW", outds["TS_DLW"])

                else:
                    if self._has(self.df[model], term):
                        outds[term] = self.df[model][term]
                        self._diag(model, term, outds[term])
                    else:
                        raise KeyError(f"Requested term '{term}' is unknown and not present in the dataset.")

            # optional sum for residual checks
            if sum_requested:
                parts = [k for k in ["TS_TURB", "TS_Qn", "TS_CRE", "TS_SAF", "TS_nSAF", "TS_DLW"] if k in outds]
                if parts:
                    acc = outds[parts[0]]
                    for k in parts[1:]:
                        a, b = self._align(acc, outds[k])
                        acc = a + b
                    outds["TS_Sum"] = acc
                    self._diag(model, "TS_Sum", outds["TS_Sum"])

            budget_dict[model] = outds

        return budget_dict
    
    def _month_day_weights(self, da: xr.DataArray) -> xr.DataArray:
        """
        Return per-sample day weights aligned to da:
          - If 'time' dim: uses time.dt.days_in_month
          - If 'month' dim: uses canonical [31,28/29,31,...] (28 for Feb by default)
        Weights return shape matches da along the time-like dimension.
        """
        if "time" in da.dims:
            return da["time"].dt.days_in_month.astype("float64")
        if "month" in da.dims:
            # canonical clim day counts (Feb=28). Adjust to 29 if you prefer leap climatology.
            day_counts = np.array([31,28,31,30,31,30,31,31,30,31,30,31], dtype="float64")
            return xr.DataArray(day_counts, dims=["month"], coords={"month": np.arange(1,13)})
        raise ValueError("Expected a 'time' or 'month' dimension.")

    def _regional_mean_ts(
            self,
            da: xr.DataArray,
            use_day_weights: bool = True,
            latmin: float | None = None,
            latmax: float | None = None,
        ) -> xr.DataArray:
        """
        Regional mean with the requested order:
          1) normalize & clip to configured lon/lat region
          2) apply land/sea mask
          3) average over longitude
          4) select latitude band
          5) weight by cos(lat) × [day weights if requested]
          6) average over latitude
        Returns a series along 'time' or 'month' (or a scalar if no time-like dim).
        """
        if "lat" not in da.dims:
            raise ValueError("Expected 'lat' dimension.")

        # 1) normalize lon and clip to configured region (safe to keep first)
        da = self._restrict_lonlat(da, self.lat_range, self.lon_range)

        # 2) mask first (aligned to current coords)
        if self.lmsk is not None:
            m = self._to_mask_like(self.lmsk, da)   # 1 where keep
            da = da.where(m > 0)

        # 3) average over longitude NOW (your requested order)
        if "lon" in da.dims:
            da = da.mean("lon", skipna=True)

        # 4) select latitude band
        if (latmin is not None) and (latmax is not None):
            da = da.where((da["lat"] >= float(latmin)) & (da["lat"] <= float(latmax)), drop=True)

        # 5) build weights: cos(lat) × [day weights if requested]
        wlat = self._generate_coslat_weight(da["lat"])         # ('lat',)
        Wlat = wlat.broadcast_like(da)                         # matches da dims (time?/month?, lat)

        if use_day_weights and ("time" in da.dims or "month" in da.dims):
            w = self._month_day_weights(da)                    # ('time',) or ('month',)
            Wtime = w.broadcast_like(da)
            W = Wlat * Wtime
        else:
            W = Wlat

        # 6) weighted average over latitude only (lon already averaged)
        if "lat" in da.dims:
            num = (da * W).sum(dim="lat", skipna=True)
            den = W.sum(dim="lat", skipna=True)
            out = num / den
        else:
            out = da  # degenerate case (no lat dim)

        # convenience coord for day weights if true time present
        if use_day_weights and ("time" in out.dims):
            out = out.assign_coords(day_weight=out["time"].dt.days_in_month.astype("float64"))

        return out

    
    def _season_selector(self, da: xr.DataArray, season: str) -> xr.DataArray:
        """
        Subset da (which has 'time' or 'month') to the requested season.
        Seasons: 'ANN','DJF','MAM','JJA','SON'
        """
        season = season.upper()
        if season == "ANN":
            return da

        if "time" in da.dims:
            mon = da["time"].dt.month
            if season == "DJF":
                return da.where((mon == 12) | (mon == 1) | (mon == 2), drop=True)
            if season == "MAM":
                return da.where((3 <= mon) & (mon <= 5), drop=True)
            if season == "JJA":
                return da.where((6 <= mon) & (mon <= 8), drop=True)
            if season == "SON":
                return da.where((9 <= mon) & (mon <= 11), drop=True)

        elif "month" in da.dims:
            if season == "DJF":
                return da.sel(month=[12, 1, 2])
            if season == "MAM":
                return da.sel(month=[3, 4, 5])
            if season == "JJA":
                return da.sel(month=[6, 7, 8])
            if season == "SON":
                return da.sel(month=[9, 10, 11])

        raise ValueError("Unknown season or missing time-like dimension.")

    def _weighted_mean_scalar(
        self, da: xr.DataArray, latmin: float = -90.0, latmax: float = -65.0
    ) -> xr.DataArray:
        """
        cos(lat) × day-weighted mean of a field with dims in {'time','month','lat'}.
        Optionally restricts to latitude band [latmin, latmax]. Returns a scalar DataArray.
        """
        if "lat" not in da.dims:
            raise ValueError("Expected 'lat' dimension.")

        # restrict latitude band
        da = da.where((da["lat"] >= latmin) & (da["lat"] <= latmax), drop=True)

        # cos(lat) weights (provided by your helper)
        wlat = self._generate_coslat_weight(da["lat"])               # dims: ('lat',)
        Wlat = wlat.broadcast_like(da)                               # dims like da

        if "time" in da.dims:
            w = self._month_day_weights(da)                          # dims: ('time',)
            Wt = w.broadcast_like(da)
            W = Wlat * Wt
            num = (da * W).sum(dim=("time", "lat"), skipna=True)
            den = W.sum(dim=("time", "lat"), skipna=True)
            return num / den

        if "month" in da.dims:
            w = self._month_day_weights(da)                          # dims: ('month',)
            Wt = w.broadcast_like(da)
            W = Wlat * Wt
            num = (da * W).sum(dim=("month", "lat"), skipna=True)
            den = W.sum(dim=("month", "lat"), skipna=True)
            return num / den

        # no time-like dimension → lat-weighted mean only
        num = (da * Wlat).sum(dim="lat", skipna=True)
        den = Wlat.sum(dim="lat", skipna=True)
        return num / den
    
    def _season_label(self, m: int) -> str:
        if m in (12, 1, 2): return "DJF"
        if m in (3, 4, 5):  return "MAM"
        if m in (6, 7, 8):  return "JJA"
        if m in (9, 10, 11):return "SON"
        raise ValueError(m)
    
    def _seasonal_agg_weighted_df(
        self,
        df_ts: pd.DataFrame,
        climatology: bool = False,
    ) -> pd.DataFrame:
        """
        Day-weighted seasonal means.

        Parameters
        ----------
        df_ts : tidy DataFrame
            ['model','term','tcoord','value'], where tcoord is datetime64 (monthly) or int month.
        climatology : bool
            - False (default): return seasonal *time series* with one row per (model, term, season, year).
                               December is shifted forward so DJF is aligned with Jan–Feb of the following year.
            - True : return *climatological* means with one row per (model, term, season),
                     ignoring years entirely.  DJF is all Dec/Jan/Feb months together.

        Returns
        -------
        tidy DataFrame with ['model','term','season','value'] (plus 'year' if climatology=False).
        """
        df = df_ts.copy()
        if df.empty:
            return pd.DataFrame(columns=["model","term","season","value"])

        # --- detect time representation ---
        is_time  = pd.api.types.is_datetime64_any_dtype(df["tcoord"])
        is_month = np.issubdtype(df["tcoord"].dtype, np.integer)

        if is_time:
            dt = pd.to_datetime(df["tcoord"])
            df["month"] = dt.dt.month
            df["year"]  = dt.dt.year
            df["days"]  = dt.dt.days_in_month.astype("float64")
        elif is_month:
            df["month"] = df["tcoord"].astype(int)
            # fallback year index: just 0
            df["year"]  = 0
            day_counts = np.array([31,28,31,30,31,30,31,31,30,31,30,31], dtype="float64")
            df["days"]  = df["month"].map({i+1: d for i, d in enumerate(day_counts)}).astype("float64")
        else:
            out = (df.groupby(["model","term"], as_index=False)
                     .agg(value=("value","mean")))
            out["season"] = "ANN"
            return out[["model","term","season","value"]]

        df["season"] = df["month"].map(self._season_label)

        if climatology:
            # --- climatology: ignore years ---
            seas = (df.assign(wv=df["value"]*df["days"])
                      .groupby(["model","term","season"], as_index=False)
                      .agg(wv=("wv","sum"), days=("days","sum")))
            seas["value"] = seas["wv"]/seas["days"]
            seas = seas[["model","term","season","value"]]

            ann = (df.assign(wv=df["value"]*df["days"])
                     .groupby(["model","term"], as_index=False)
                     .agg(wv=("wv","sum"), days=("days","sum")))
            ann["value"] = ann["wv"]/ann["days"]
            ann["season"] = "ANN"
            ann = ann[["model","term","season","value"]]

            out = pd.concat([seas, ann], ignore_index=True)

        else:
            # --- seasonal series with year info (shift Dec forward) ---
            df.loc[df["month"] == 12, "year"] += 1

            seas = (df.assign(wv=df["value"]*df["days"])
                      .groupby(["model","term","season","year"], as_index=False)
                      .agg(wv=("wv","sum"), days=("days","sum")))
            seas["value"] = seas["wv"]/seas["days"]
            seas = seas[["model","term","season","year","value"]]

            ann = (df.assign(wv=df["value"]*df["days"])
                     .groupby(["model","term","year"], as_index=False)
                     .agg(wv=("wv","sum"), days=("days","sum")))
            ann["value"] = ann["wv"]/ann["days"]
            ann["season"] = "ANN"
            ann = ann[["model","term","season","year","value"]]

            out = pd.concat([seas, ann], ignore_index=True)

        # ordered season category
        cats = ["DJF","MAM","JJA","SON","ANN"]
        out["season"] = out["season"].astype(pd.CategoricalDtype(categories=cats, ordered=True))
        return out.sort_values(["model","term","season"]).reset_index(drop=True)


    def _safe(self, s: str) -> str:
        return "".join(c if c.isalnum() or c in ("-", "_", ".") else "_" for c in s)

    def export_seasonal_means_csv(
        self,
        *,
        latmin=None,
        latmax=None,
        include_residual=True,
        include_flux_anoms=True,
        include_flux_refs=True,
        outdir=".",
        filename="seasonal_budget_means_{budget_type}_{region}.csv",
        split_by_model=False,
        split_by_model_season=False,  # NEW: one CSV per model × season
        filename_tmpl="seasonal_budget_means_{budget_type}_{model}_{region}.csv",
        filename_tmpl_season="seasonal_budget_means_{budget_type}_{model}_{season}_{region}.csv",  # NEW
        model_key_to_name=None,
        seasonal_weighting: str = "day",   # <--- NEW
    ) -> list[str]:
        """
        Compute seasonal/ANN means and write to CSV(s).

        Modes:
          - Default (both splits False): single tidy CSV with all models & seasons.
          - split_by_model=True: one tidy CSV per model (all seasons).
          - split_by_model_season=True: one tidy CSV per model × season.
            (This takes precedence over split_by_model when both are True.)

        Returns:
          list[str]: paths to the written CSV files.
        """

        os.makedirs(outdir, exist_ok=True)

        # Build monthly series with requested terms
        df_ts = self.regional_budget_timeseries(
            use_day_weights=True,
            compute_residual=include_residual,
            include_flux_anoms=include_flux_anoms,
            include_flux_refs=include_flux_refs,
            latmin=latmin, latmax=latmax,    # pass the band
        )
        
        # Use weighted, DJF-aware seasonal means
        df_seas = self._seasonal_agg_weighted_df(df_ts, climatology=True).copy()

        # Stable season ordering
        season_order = pd.CategoricalDtype(categories=["DJF","MAM","JJA","SON","ANN"], ordered=True)
        if "season" in df_seas.columns:
            df_seas["season"] = df_seas["season"].astype(season_order)

        written: list[str] = []

        # --- one file per model × season (highest precedence) ---
        if split_by_model_season:
            for model in sorted(df_seas["model"].unique()):
                disp_model = model_key_to_name(model) if callable(model_key_to_name) else model
                for season in [s for s in season_order.categories if s in set(df_seas["season"])]:
                    sub = df_seas[(df_seas["model"] == model) & (df_seas["season"] == season)].copy()
                    if sub.empty:
                        continue
                    fname = filename_tmpl_season.format(
                        budget_type=self.budget_type,
                        model=self._safe(str(disp_model)),
                        season=str(season),
                        region=self._safe(str(self.regnam)),
                    )
                    fpath = os.path.join(outdir, fname)
                    sub.to_csv(fpath, index=False)
                    written.append(fpath)
            return written

        # --- one file per model ---
        if split_by_model:
            for model in sorted(df_seas["model"].unique()):
                sub = df_seas[df_seas["model"] == model].copy()
                disp_model = model_key_to_name(model) if callable(model_key_to_name) else model
                fname = filename_tmpl.format(
                    budget_type=self.budget_type,
                    model=self._safe(str(disp_model)),
                    region=self._safe(str(self.regnam)),
                )
                fpath = os.path.join(outdir, fname)
                sub.to_csv(fpath, index=False)
                written.append(fpath)
            return written

        # --- single file with all models ---
        fname = filename.format(
            budget_type=self.budget_type,
            region=self._safe(str(self.regnam))
        )
        fpath = os.path.join(outdir, fname)
        df_seas.to_csv(fpath, index=False)
        written.append(fpath)
        return written
    
    @staticmethod
    def _to_pandas_datetimes(tvals) -> np.ndarray:
        """Convert an array/Series of time values (possibly cftime) to pandas-friendly datetimes."""
        import xarray as xr
        try:
            # Works for cftime arrays and returns numpy datetime64[ns]
            return xr.coding.cftimeindex.CFTimeIndex(np.asarray(tvals)).to_datetimeindex().values
        except Exception:
            # already datetime64 / python datetime
            return pd.to_datetime(np.asarray(tvals), errors="raise").values

    # ---------- public API ----------
    def regional_budget_timeseries(
        self,
        use_day_weights=True,
        compute_residual=True,
        include_flux_anoms=False,
        include_flux_refs=False,
        *,
        latmin: float | None = None,
        latmax: float | None = None,
    ):
        """
        Returns tidy DataFrame ['model','term','tcoord','value'].
        - TS_* (K) always available per `budget_vars`.
        - If include_flux_anoms: include d* and *_flux (W m^-2).
        - If include_flux_refs:  include *_m and *_r (W m^-2 absolute).
        """
        budget_dict = (
            self.budget_fn(self.budget_type, self.model_list, {t: {} for t in self.plot_terms}, self.df)
            if callable(self.budget_fn)
            else self._derive_energy_budget_internal()
        )
        flux_dict = self._derive_flux_anomalies_internal() if (include_flux_anoms or include_flux_refs) else {}

        rows = []
        for model in self.model_list:
            # find a sample to infer t-dim
            sample_da = None
            for t in self.plot_terms:
                if (model in budget_dict) and (t in budget_dict[model]):
                    sample_da = budget_dict[model][t]; break
            if (sample_da is None) and (model in flux_dict) and (len(flux_dict[model].data_vars) > 0):
                sample_da = next(iter(flux_dict[model].data_vars.values()))
            if sample_da is None:
                continue

            tdim = "time" if "time" in sample_da.dims else ("month" if "month" in sample_da.dims else None)
            regional: dict[str, xr.DataArray] = {}

            # TS_* (K)
            for term in self.plot_terms:
                if term not in budget_dict[model]:
                    continue
                ts = self._regional_mean_ts(
                    budget_dict[model][term],
                    use_day_weights=use_day_weights,
                    latmin=latmin, latmax=latmax,
                )
                regional[term] = ts

            if compute_residual and ("TS_ALL" in regional):
                parts = [k for k in ["TS_Qn","TS_TURB","TS_CRE","TS_SAF","TS_nSAF","TS_DLW"] if k in regional]
                if parts:
                    acc = regional[parts[0]]
                    for k in parts[1:]:
                        a, b = self._align(acc, regional[k]); acc = a + b
                    all_aligned, sum_aligned = self._align(regional["TS_ALL"], acc)
                    regional["TS_RESID"] = all_aligned - sum_aligned

            # Flux fields (W m^-2)
            if model in flux_dict:
                for vname, da in flux_dict[model].data_vars.items():
                    is_anom = vname.startswith("d") or vname.endswith("_flux")
                    is_ref  = vname.endswith("_m") or vname.endswith("_r")
                    if (is_anom and include_flux_anoms) or (is_ref and include_flux_refs):
                        ts = self._regional_mean_ts(
                            da,
                            use_day_weights=use_day_weights,
                            latmin=latmin, latmax=latmax,
                        )
                        regional[vname] = ts

            # emit tidy
            for term, ts in regional.items():
                if tdim is None:
                    rows.append({"model": model, "term": term, "tcoord": np.nan, "value": float(ts.values)})
                else:
                    tvals = ts[tdim].values
                    vvals = ts.values
                    if tdim == "time":
                        tvals = self._to_pandas_datetimes(tvals)
                        rows.extend({"model": model, "term": term, "tcoord": t, "value": float(v)}
                                    for t, v in zip(tvals, vvals))
                    else:  # 'month'
                        rows.extend({"model": model, "term": term, "tcoord": int(t), "value": float(v)}
                                    for t, v in zip(tvals, vvals))

        df_ts = pd.DataFrame(rows)
        df_ts.attrs = {"region": self.regnam, "lat_range": self.lat_range, "lon_range": self.lon_range}
        return df_ts

    def _unit_for(self,term: str) -> str:
        # Temperature contributions are K; raw flux anoms/refs are W m^-2
        if term.startswith("TS_"):
            return "K"
        return "W m$^{-2}$"
    
    def plot_regional_timeseries(
        self,
        latmin=None,
        latmax=None,
        terms=None,
        show=False,
        outdir=".",
        filename=None,
        include_residual=True,
        legend_outside=True,
        fontz=11
    ):
        """
        Line plots of regional-mean time series.
        - One panel per model; multiple terms per panel.
        - If include_residual=True, plot TS_RESID (if available).
        Returns the output path.
        """

        df_ts = self.regional_budget_timeseries(use_day_weights=True, compute_residual=include_residual)

        if terms is None:
            all_terms = [t for t in self.plot_terms if t != "TS_Sum"]
            if include_residual and ("TS_RESID" in df_ts["term"].unique()):
                all_terms = all_terms + ["TS_RESID"]
            terms = all_terms

        models = list(dict.fromkeys(df_ts["model"]))
        nrows, ncols = len(models), 1
        fig_h = max(2.8 * nrows, 4.0)
        fig_w = 12.0

        fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False, sharex=True)
        for i, model in enumerate(models):
            ax = axes[i, 0]
            sub = df_ts[df_ts["model"] == model]

            # plot each term
            for term in terms:
                ss = sub[sub["term"] == term]
                if ss.empty:
                    continue
                x = ss["tcoord"]
                y = ss["value"]
                ax.plot(x, y, label=term, linewidth=1.8)

            units = { self._unit_for(t) for t in terms if t in set(sub["term"]) }
            ylabel = ", ".join(sorted(units)) if units else "K"
            ax.set_ylabel(ylabel, fontsize=fontz)
            title = self._meta_get(self.run_dict.get(model, {}), "name", str(model))
            ax.set_title(f"{title} — {self.regnam}", fontsize=fontz+1, fontweight="bold")
            ax.grid(True, alpha=0.35)
            ax.tick_params(labelsize=fontz*0.95)

            if legend_outside:
                ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=fontz*0.9)
            else:
                ax.legend(fontsize=fontz*0.9)

        plt.tight_layout()
        if filename is None:
            main_var = (terms[0] if terms else "budget")
            filename = f"regional_budget_ts_{self.budget_type}_{main_var}_{self.regnam}.pdf"
        outpath = os.path.join(outdir, filename)
        fig.savefig(outpath, bbox_inches="tight")
        if show:
            plt.show()
        plt.close(fig)
        return outpath

In [ ]:
if __name__ == "__main__":
    # --- region settings (config/regions.json) ---
    from regions import region_catalogs, region_settings

    RS = region_settings(REGION)
    RUN_CATALOG, VARIABLE_CATALOG = region_catalogs(REGION)

    # --- paths ---
    data_path = CLIMO_DIR
    out_path  = DIAG_DIR
    fig_path  = FIG_DIR

    # --- controls ---
    regnam     = REGION
    dtype      = "climo"
    latmin, latmax = RS["budget_lat_band"] or RS["lat_range"]
    
    # --- echo metadata safely ---
    refexp = RS["refexp"]
    run_info = RUN_CATALOG[refexp]
    try:
        ref_name = run_info.name
        ref_period = run_info.period
    except AttributeError:
        ref_name = run_info["name"]
        ref_period = run_info["period"]

    reg_info = REGION_CATALOG[regnam]
    try:
        (lat_min, lat_max) = reg_info.lat_range
        (lon_min, lon_max) = reg_info.lon_range
    except AttributeError:
        (lat_min, lat_max), (lon_min, lon_max) = reg_info

    print(f"reference data and period: {ref_name}, {ref_period}")
    print(f"processed region: {regnam}, lat=({lat_min},{lat_max}), lon=({lon_min},{lon_max})")

    out_dir = f"{out_path}/{regnam}"
    fig_dir = f"{fig_path}/{regnam}"
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(fig_dir, exist_ok=True)

    # --- read regionalized datasets ---
    reader = ExpDataReader(
        data_dir=data_path,
        regnam=regnam,
        dtype=dtype,
        verbose=True
    )
    run_catalog, exp_dict, land_mask = reader.read()

    print("Loaded runs:")
    pprint.pprint(list(run_catalog.keys()))

    # models to include (exclude pure references and other unwanted runs)
    model_list = [k for k in run_catalog if k != refexp]
    for exclude in [
        "MODIS.observation",
        "CERES_EBAF.observation",
        "MERRA2.analysis",
        "ERA5.analysis",
        "ERA5.ensmn",
        "v3.LR.piControl.hexagonal",
        "20250327.v3.SORRME3r3.piControl.alfred2.chrysalis",
    ]:
        if exclude in model_list:
            model_list.remove(exclude)

    # --- budget controls ---
    budget_vars  = list(Budget_CATALOG.keys())   # e.g., ["TS_ALL","TS_TURB",...,"TS_Sum"]
    region_bounds = ((lat_min, lat_max), (lon_min, lon_max))
    fontz = 28

    # normalize unicode dashes in period for filenames
    save_period = (
        ref_period.replace("\u2010","-").replace("\u2011","-").replace("\u2012","-")
                  .replace("\u2013","-").replace("\u2014","-").replace("\u2212","-")
    )

    # --- loop over budget_type and make plots ---
    for budget_type in ["best_obs", "all_merra"]:

        rb = RegionalMeanBudget(
            run_dict=run_catalog,
            df=exp_dict,
            lmsk=land_mask,
            regnam=regnam,
            model_list=model_list,
            budget_vars=budget_vars,
            term_specs=Budget_CATALOG,   # optional; units/names not required here
            budget_type=budget_type,
            region_bounds=region_bounds,
            debug=False,
        )

        # 1) monthly time series
        df_ts = rb.regional_budget_timeseries(
            use_day_weights=True,
            compute_residual=True,
            include_flux_anoms=True,   # was: include_fluxes=True
            include_flux_refs=True     # add this
        )

        # 2) seasonal CSV
        seas_csv = rb.export_seasonal_means_csv(
            latmin=latmin, latmax=latmax,
            include_residual=True,
            include_flux_anoms=True,
            include_flux_refs=True,
            outdir=out_dir,
            split_by_model_season=True,  # NEW: per model × season
            filename_tmpl_season=(
                f"seasonal_budget_means_{budget_type}_"  # prefix
                f"{{model}}_{{season}}_"                  # placeholders filled inside method
                f"{rb._safe(regnam)}_{rb._safe(str(save_period))}.csv"
            ),
        )
        print("[ok] seasonal CSVs (per model × season):")
        for p in seas_csv:
            print("  ", p)
    
        print(f"[ok] seasonal CSV (tidy): {seas_csv}")

        # Build a consistent filename and write directly to fig_dir
        outfile = f"regional_budget_ts_{budget_type}_{regnam}_{save_period}.pdf"
        savepath = os.path.join(fig_dir, outfile)

        try:
            rb.plot_regional_timeseries(
                terms=None,                  # default = all requested terms (+ TS_RESID if available)
                include_residual=True,
                legend_outside=True,
                fontz=fontz,
                outdir=fig_dir,
                filename=outfile,
                show=False
            )
            print(f"[ok] saved: {savepath}")
        except Exception as e:
            print(f"[fail] regional_mean_budget: {e}")
            